#### Modeling

Setup and load data

In [2]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import math
import statsmodels.api as sm  

In [4]:
# load data
X_train = pd.read_csv("../data/X_train_movies.csv")
X_test  = pd.read_csv("../data/X_test_movies.csv")

y_train = pd.read_csv("../data/y_train_movies.csv")
y_test  = pd.read_csv("../data/y_test_movies.csv")

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(8204, 26) (6090, 26) (8204, 1) (6090, 1)


Define evaluation function

In [5]:
def score_regression(y_true, y_pred):
    """
    Return R^2, MAE, and RMSE for a regression model.
    """
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    
    return pd.Series({"R2": r2, "MAE": mae, "RMSE": rmse})

Fit 3 linear regression models on the training data:

• Model 1: Use only runtime_in_minutes

• Model 2: Use runtime_in_minutes and kid_friendly

• Model 3: Use runtime_in_minutes, kid_friendly and the dummy columns for the genres

Choose feature sets for the three models

In [6]:
# Model 1: runtime only
feat_model1 = ["runtime_in_minutes"]

# Model 2: runtime + kid_friendly
feat_model2 = ["runtime_in_minutes", "kid_friendly"]

# Model 3: runtime + kid_friendly + all genre dummies
exclude_from_genre = [
    "runtime_in_minutes",
    "kid_friendly",
    "kid_family",
    "days_to_stream",
    "holiday_blockbuster",
]

genre_cols = [c for c in X_train.columns if c not in exclude_from_genre]

feat_model3 = ["runtime_in_minutes", "kid_friendly"] + genre_cols

len(genre_cols), genre_cols[:10]  # just to sanity-check

(21,
 ['Animation',
  'Anime & Manga',
  'Art House & International',
  'Classics',
  'Comedy',
  'Cult Movies',
  'Documentary',
  'Drama',
  'Faith & Spirituality',
  'Gay & Lesbian'])

Fit the three linear models and score them

In [8]:
results = {}

# Model 1
m1 = LinearRegression()
m1.fit(X_train[feat_model1], y_train)
y_pred1 = m1.predict(X_test[feat_model1])
results["Model 1: runtime only"] = score_regression(y_test, y_pred1)

# Model 2 
m2 = LinearRegression()
m2.fit(X_train[feat_model2], y_train)
y_pred2 = m2.predict(X_test[feat_model2])
results["Model 2: runtime + kid_friendly"] = score_regression(y_test, y_pred2)

# Model 3 
m3 = LinearRegression()
m3.fit(X_train[feat_model3], y_train)
y_pred3 = m3.predict(X_test[feat_model3])
results["Model 3: runtime + kid_friendly + genres"] = score_regression(y_test, y_pred3)

# all metrics into one DataFrame
model_scores = (
    pd.DataFrame(results)
    .T  # models as rows
)

model_scores

,R2,MAE,RMSE
Model 1: runtime only,-0.023767,25.071866,28.711103
Model 2: runtime + kid_friendly,-0.025618,25.105363,28.737037
Model 3: runtime + kid_friendly + genres,0.030971,24.001836,27.933010
